In [1]:
import torch
import torch.nn as nn


In [4]:
#Mini VIT
class PatchEmbedding(nn.Module):
    def __init__(self,img_size=128,patch_size=16,in_channels=1,embed_dim=64):
        super(PatchEmbedding,self).__init__()
        self.patch_size = patch_size
        self.num_patches = (img_size//patch_size)**2
        self.proj = nn.Conv2d(in_channels,embed_dim,kernel_size=patch_size,stride=patch_size)
    def forward(self,x):
        x=self.proj(x)
        x=x.flatten(2)
        x=x.transpose(1,2)
        return x
        
class PositionalEncoding(nn.Module):
    def __init__(self,num_patches,embed_dim):
        super(PositionalEncoding,self).__init__()
        self.pos_embedding = nn.Parameter(torch.zeros(1,num_patches,embed_dim))
    def forward(self,x):
        return x + self.pos_embedding


class TransformerBlock(nn.Module):
    def __init__(self,embed_dim,num_heads,mlp_dim,dropout=0.1):
        super(TransformerBlock,self).__init__()
        self.attn = nn.MultiheadAttention(embed_dim,num_heads,dropout=dropout)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim,mlp_dim),
            nn.ReLU(),
            nn.Linear(mlp_dim,embed_dim)
        )
        self.norm2 = nn.LayerNorm(embed_dim)
    def forward(self,x):
        attn_output,_ =self.attn(x,x,x)
        x=self.norm1(x+attn_output)
        mlp_output =self.mlp(x)
        x= self.norm2(x+mlp_output)
        return x

class MiniVit(nn.Module):
    def __init__(self,img_size=128,patch_size=16,in_channels=3,embed_dim=64,num_heads=4,mlp_dim=128,num_classes=4,num_blocks=4):
        super(MiniVit,self).__init__()
        self.patch_embed = PatchEmbedding(img_size,patch_size,in_channels,embed_dim)
        self.pos_embed = PositionalEncoding(self.patch_embed.num_patches,embed_dim)
        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim,num_heads,mlp_dim) for _ in range(num_blocks)
        ])
        self.classifier = nn.Linear(embed_dim,num_classes)
    def forward(self,x):
        x=self.patch_embed(x)
        x=self.pos_embed(x)
        for block in self.blocks:
            x=block(x)
        x=x.mean(dim=1)
        x=self.classifier(x)
        return x

model =MiniVit()
print(model)
print(model(torch.randn( 10,3, 128, 128)))

MiniVit(
  (patch_embed): PatchEmbedding(
    (proj): Conv2d(3, 64, kernel_size=(16, 16), stride=(16, 16))
  )
  (pos_embed): PositionalEncoding()
  (blocks): ModuleList(
    (0-3): 4 x TransformerBlock(
      (attn): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
      )
      (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (mlp): Sequential(
        (0): Linear(in_features=64, out_features=128, bias=True)
        (1): ReLU()
        (2): Linear(in_features=128, out_features=64, bias=True)
      )
      (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
    )
  )
  (classifier): Linear(in_features=64, out_features=4, bias=True)
)
tensor([[ 0.1846,  0.3619, -0.0806,  0.0787],
        [ 0.3014,  0.3550, -0.1011,  0.0660],
        [ 0.1818,  0.2217,  0.0194,  0.0682],
        [ 0.1380,  0.2301,  0.0604,  0.0259],
        [ 0.2199,  0.3184, -0.1578,  0.0406],
        [ 0.1998,  0.2290, -0.0